In [1]:
import numpy as np
import nibabel as nib
from skimage.morphology import skeletonize
from skimage.measure import label
from catheter_segmentation.Unet_model import UNet3D
import torch
import ants

In [2]:
subject_number = '002'
subject_path = f'data/Train/subj{subject_number}'
img_path = subject_path + f'/subj{subject_number}_image.nii.gz'
art_mask_path = subject_path + f'/subj{subject_number}_artifact.nii.gz'
cat_mask_path = subject_path + f'/subj{subject_number}_catheter.nii.gz'

# Load masks
img = nib.load(img_path)
catheter_mask = nib.load(cat_mask_path)
artifact_mask = nib.load(art_mask_path)

catheter_data = catheter_mask.get_fdata() > 0.5
artifact_data = artifact_mask.get_fdata() > 0.5


In [3]:
def sliding_window_inference(model, volume, patch_size=64, stride=32, device='cuda', threshold=0.5):
    model.eval()
    D, H, W = volume.shape
    p = patch_size

    pred_sum = np.zeros((D, H, W), dtype=np.float32)
    count_map = np.zeros((D, H, W), dtype=np.float32)

    # Normalize volume
    vol_norm = (volume - volume.min()) / (volume.max() - volume.min() + 1e-8)
    vol_tensor = torch.tensor(vol_norm).unsqueeze(0).unsqueeze(0).float()  # (1,1,D,H,W)

    with torch.no_grad():
        for z in range(0, D - p + 1, stride):
            for y in range(0, H - p + 1, stride):
                for x in range(0, W - p + 1, stride):
                    patch = vol_tensor[:, :, z:z+p, y:y+p, x:x+p].to(device)
                    logit = model(patch)
                    prob = torch.sigmoid(logit).squeeze().cpu().numpy()
                    pred_sum[z:z+p, y:y+p, x:x+p] += prob
                    count_map[z:z+p, y:y+p, x:x+p] += 1

    # Average overlapping predictions
    count_map = np.maximum(count_map, 1)
    avg_pred = pred_sum / count_map
    return (avg_pred > threshold).astype(np.uint8)

device = 'cpu'
model = UNet3D(base_channels=16).to(device)

model_path = './catheter_segmentation/catheter_seg_model.pth'
checkpoint = torch.load(model_path, map_location=device)
state_dict = checkpoint['model_state_dict']
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.load_state_dict(state_dict)
model.to(device)
model.eval()

UNet3D(
  (enc1): ConvBlock3D(
    (block): Sequential(
      (0): Conv3d(1, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      (1): InstanceNorm3d(16, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
      (2): ReLU(inplace=True)
      (3): Conv3d(16, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      (4): InstanceNorm3d(16, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
      (5): ReLU(inplace=True)
    )
  )
  (enc2): ConvBlock3D(
    (block): Sequential(
      (0): Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      (1): InstanceNorm3d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
      (2): ReLU(inplace=True)
      (3): Conv3d(32, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      (4): InstanceNorm3d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
      (5): ReLU(inplace=True)
    )
  )
  (enc3): ConvBlock3D(
    (b

In [4]:
predicted_catheter_mask = sliding_window_inference(model, img.get_fdata(), patch_size=64, stride=32, device=device)

In [34]:
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path
from skimage.morphology import skeletonize
from scipy.ndimage import binary_dilation, binary_erosion, label as scipy_label
import torch
from monai.networks.nets import UNet
import skimage.transform as skT

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
CONFIG = {
    "subject_path"   : "./data/Train/subj",
    "subject_number" : "002",
    "model_path"     : "./checkpoints/best_model.pth",
    "output_dir"     : Path("./outputs/length_estimation"),
}
CONFIG["output_dir"].mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────
def load_subject(subject_path, subject_number):
    base   = f"{subject_path}{subject_number}/subj{subject_number}"
    volume = nib.load(f"{base}_image.nii.gz")
    data   = volume.get_fdata()

    # voxel spacing in mm (x, y, z)
    voxel_spacing = np.array(volume.header.get_zooms()[:3])

    catheter_mask = nib.load(f"{base}_catheter.nii.gz").get_fdata() > 0.5
    artifact_mask = nib.load(f"{base}_artifact.nii.gz").get_fdata() > 0.5

    print(f"Volume shape     : {data.shape}")
    print(f"Voxel spacing mm : {voxel_spacing}")
    print(f"Catheter voxels  : {catheter_mask.sum()}")
    print(f"Artifact voxels  : {artifact_mask.sum()}")

    return data, catheter_mask, artifact_mask, voxel_spacing, volume.affine

def skeletonize_mask(mask):
    """Thin a binary 3D mask to a 1-voxel centerline."""
    return skeletonize(mask.astype(bool))

def find_endpoints(skeleton):
    """Find tip and valve end — voxels with exactly 1 neighbor in skeleton."""
    from scipy.ndimage import convolve
    kernel       = np.ones((3, 3, 3), dtype=int)
    neighbor_cnt = convolve(skeleton.astype(int), kernel) * skeleton.astype(int)
    # endpoint = self (1) + exactly 1 neighbor = value of 2
    endpoints = np.argwhere(neighbor_cnt == 2)
    return endpoints

def trace_path(points):
    """Nearest-neighbor traversal to order skeleton voxels into a path."""
    if len(points) == 0:
        return points
    unvisited = list(range(len(points)))
    path      = [unvisited.pop(0)]
    while unvisited:
        last    = points[path[-1]]
        dists   = [np.linalg.norm(points[i] - last) for i in unvisited]
        nearest = unvisited[int(np.argmin(dists))]
        path.append(nearest)
        unvisited.remove(nearest)
    return np.array([points[i] for i in path])

def compute_path_length(skeleton, voxel_spacing):
    """Convert skeleton to physical path length in mm."""
    coords      = np.argwhere(skeleton)
    if len(coords) < 2:
        return 0.0, np.array([])
    phys_coords = coords * voxel_spacing
    ordered     = trace_path(phys_coords)
    diffs       = np.diff(ordered, axis=0)
    seg_lengths = np.linalg.norm(diffs, axis=1)
    return float(np.sum(seg_lengths)), ordered

def traditional_approach(catheter_mask, artifact_mask, voxel_spacing):
    # 1. Skeletonize ONLY the catheter (not artifact)
    catheter_skel = skeletonize_mask(catheter_mask)
    print(f"  Catheter skeleton voxels: {catheter_skel.sum()}")

    # 2. Find the endpoint of catheter skeleton closest to artifact
    #    This is the point where catheter enters the artifact blob
    skel_coords = np.argwhere(catheter_skel)
    art_coords  = np.argwhere(artifact_mask)

    # For each skeleton endpoint find distance to artifact
    endpoints = find_endpoints(catheter_skel)
    print(f"  Catheter endpoints: {len(endpoints)}")

    # Find which endpoint is closest to artifact (valve end)
    min_dist   = np.inf
    valve_entry = None
    for ep in endpoints:
        dists = np.linalg.norm(art_coords - ep, axis=1)
        d     = dists.min()
        if d < min_dist:
            min_dist    = d
            valve_entry = ep
            valve_exit  = art_coords[np.argmin(dists)]  # closest artifact point

    print(f"  Catheter enters artifact at voxel : {valve_entry}")
    print(f"  Closest artifact point            : {valve_exit}")
    print(f"  Gap between them (voxels)         : {min_dist:.1f}")

    # 3. Compute visible catheter length (skeleton only)
    visible_len, ordered_path = compute_path_length(catheter_skel, voxel_spacing)
    print(f"  Visible length: {visible_len:.2f} mm")

    # 4. Compute occluded length
    #    = straight line from catheter tip into artifact to artifact centroid
    #      (best estimate since we can't see inside the blob)
    art_centroid  = art_coords.mean(axis=0)             # centre of blob
    entry_phys    = valve_entry * voxel_spacing
    centroid_phys = art_centroid * voxel_spacing
    occluded_len  = float(np.linalg.norm(centroid_phys - entry_phys))
    print(f"  Occluded length (entry → artifact centroid): {occluded_len:.2f} mm")

    total_len = visible_len + occluded_len
    print(f"\n  Visible catheter     : {visible_len:.2f} mm")
    print(f"  Occluded by artifact : {occluded_len:.2f} mm  ← estimated")
    print(f"  TOTAL LENGTH         : {total_len:.2f} mm  ({total_len/10:.2f} cm)")

    return {
        "method"       : "Traditional",
        "total_mm"     : total_len,
        "visible_mm"   : visible_len,
        "occluded_mm"  : occluded_len,
        "skeleton"     : catheter_skel,
        "ordered_path" : ordered_path,
        "valve_entry"  : valve_entry,
        "valve_exit"   : valve_exit,
        "art_centroid" : art_centroid,
    }

def load_model(model_path):
    model = UNet(
        spatial_dims=3,
        in_channels=2,
        out_channels=1,
        channels=(32, 64, 128, 256),
        strides=(2, 2, 2),
        num_res_units=2,
    ).to(DEVICE)
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.eval()
    print(f"  Model loaded from {model_path}")
    return model

def predict_mask(model, image, estimation, voxel_spacing, original_shape):
    """Run model on full volume using sliding window over z-slices."""
    # Crop around centre of volume (same as training)
    cx = image.shape[0] // 2
    cy = image.shape[1] // 2
    cz = image.shape[2] // 2

    # Resize to model input size
    img_resized  = skT.resize(image,      (96, 96, 96), order=1,
                               preserve_range=True, anti_aliasing=False)
    est_resized  = skT.resize(estimation, (96, 96, 96), order=1,
                               preserve_range=True, anti_aliasing=False)

    x = np.stack([img_resized, est_resized], axis=0).astype(np.float32)
    x = torch.from_numpy(x).unsqueeze(0).to(DEVICE)   # (1, 2, 96, 96, 96)

    with torch.no_grad():
        pred = torch.sigmoid(model(x))                 # (1, 1, 96, 96, 96)

    pred_np = pred.squeeze().cpu().numpy()             # (96, 96, 96)

    # Resize prediction back to original shape
    pred_full = skT.resize(pred_np, original_shape, order=1,
                            preserve_range=True, anti_aliasing=False)
    pred_mask = (pred_full > 0.5).astype(bool)
    return pred_mask

def model_approach(image, estimation, artifact_mask, voxel_spacing, model_path):
    print("\n" + "="*55)
    print("APPROACH B — Model-based (UNet prediction)")
    print("="*55)

    # 1. Load model and predict
    model         = load_model(model_path)
    original_shape = image.shape
    pred_mask     = predict_mask(model, image, estimation,
                                  voxel_spacing, original_shape)
    print(f"  Predicted mask voxels : {pred_mask.sum()}")

    # 2. Combine predicted mask + artifact region
    combined = pred_mask | artifact_mask
    print(f"  Combined mask voxels  : {combined.sum()}")

    # 3. Fill and skeletonize
    combined_filled = binary_dilation(combined, iterations=2)
    combined_filled = binary_erosion(combined_filled, iterations=2)
    skeleton        = skeletonize_mask(combined_filled)
    print(f"  Skeleton voxels       : {skeleton.sum()}")

    # 4. Keep largest component
    labeled, n = scipy_label(skeleton)
    if n > 1:
        sizes    = [np.sum(labeled == i) for i in range(1, n+1)]
        skeleton = (labeled == (np.argmax(sizes) + 1))
        print(f"  Kept largest of {n} skeleton components")

    # 5. Endpoints
    endpoints = find_endpoints(skeleton)
    print(f"  Endpoints found       : {len(endpoints)}")

    # 6. Path length
    length_mm, ordered_path = compute_path_length(skeleton, voxel_spacing)

    # Breakdown
    pred_skel      = skeletonize_mask(pred_mask)
    visible_len, _ = compute_path_length(pred_skel, voxel_spacing)
    occluded_len   = length_mm - visible_len

    print(f"\n  Visible catheter      : {visible_len:.2f} mm")
    print(f"  Occluded by artifact  : {occluded_len:.2f} mm  ← recovered")
    print(f"  TOTAL LENGTH          : {length_mm:.2f} mm  ({length_mm/10:.2f} cm)")

    return {
        "method"        : "Model-based",
        "total_mm"      : length_mm,
        "visible_mm"    : visible_len,
        "occluded_mm"   : occluded_len,
        "skeleton"      : skeleton,
        "ordered_path"  : ordered_path,
        "pred_mask"     : pred_mask,
        "combined_mask" : combined,
    }

def visualise(image, catheter_mask, artifact_mask, results, output_dir, subject_number):
    fig, axes = plt.subplots(len(results), 4, figsize=(20, 6*len(results)))
    if len(results) == 1:
        axes = axes[np.newaxis]

    for row, res in enumerate(results):
        # pick slice with most skeleton voxels
        skel      = res["skeleton"]
        best_z    = int(np.argmax(skel.sum(axis=(0,1))))
        path      = res["ordered_path"]

        def show(ax, data, title, cmap="gray", vmin=None, vmax=None):
            ax.imshow(data[:, :, best_z].T, origin="lower",
                      cmap=cmap, vmin=vmin, vmax=vmax)
            ax.set_title(f"[{res['method']}]\n{title}", fontsize=9)
            ax.axis("off")

        show(axes[row, 0], image,                          "MRI Image")
        show(axes[row, 1], catheter_mask.astype(float),   "Catheter Mask",  cmap="hot")
        show(axes[row, 2], artifact_mask.astype(float),   "Artifact Mask",  cmap="hot")
        show(axes[row, 3], skel.astype(float),            "Centerline",     cmap="hot")

        # overlay path on MRI
        axes[row, 0].set_title(
            f"[{res['method']}]  Total={res['total_mm']:.1f}mm  "
            f"Visible={res['visible_mm']:.1f}mm  "
            f"Occluded={res['occluded_mm']:.1f}mm", fontsize=8
        )

    plt.tight_layout()
    out = output_dir / f"subj{subject_number}_length_estimation.png"
    plt.savefig(str(out), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\nFigure saved: {out}")

def print_summary(results):
    print("\n" + "="*55)
    print("SUMMARY COMPARISON")
    print("="*55)
    print(f"{'Method':<20} {'Total (mm)':>12} {'Total (cm)':>12} {'Visible':>10} {'Occluded':>10}")
    print("-"*55)
    for r in results:
        print(f"{r['method']:<20} {r['total_mm']:>12.2f} "
              f"{r['total_mm']/10:>12.2f} "
              f"{r['visible_mm']:>10.2f} "
              f"{r['occluded_mm']:>10.2f}")

    if len(results) == 2:
        diff = abs(results[0]["total_mm"] - results[1]["total_mm"])
        print(f"\n  Difference between methods: {diff:.2f} mm")

In [16]:
print(img.header.get_zooms())

(np.float32(1.0), np.float32(1.0), np.float32(1.0))


In [19]:
# ================================================================
# Catheter Length Estimation — improved, less underestimation
# Assumes predicted_catheter_mask and artifact_mask (artifact_data)
# are already loaded as numpy arrays in memory.
# ================================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from scipy.ndimage import binary_fill_holes, label, distance_transform_edt
from skimage.morphology import skeletonize
from skimage.measure import regionprops
import networkx as nx
import warnings

PIXEL_SPACING_MM = 1.0   # <-- update to your actual mm/px


# ── helpers ──────────────────────────────────────────────────────────────────

def to_2d_bool(arr, name="mask"):
    arr = np.asarray(arr).squeeze()
    if arr.ndim == 3:
        best = int((arr > 0).sum(axis=(0, 1)).argmax())
        print(f"[warn] '{name}' is 3-D; using slice {best}")
        arr = arr[..., best]
    arr = arr.astype(float)
    return (arr / arr.max()) > 0.5

def keep_largest(mask):
    lbl, n = label(mask)
    if n == 0: return mask
    sizes = [(lbl == i).sum() for i in range(1, n + 1)]
    return lbl == (np.argmax(sizes) + 1)

def build_graph(skel):
    G   = nx.Graph()
    pts = set(map(tuple, np.argwhere(skel).tolist()))
    for r, c in pts:
        G.add_node((r, c))
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                if dr == dc == 0: continue
                nb = (r + dr, c + dc)
                if nb in pts:
                    G.add_edge((r, c), nb, weight=np.sqrt(dr**2 + dc**2))
    return G

def get_endpoints(G):
    return [n for n, d in G.degree() if d == 1]

def trace_longest(G, eps):
    best_l, best_p = -1, []
    for i, src in enumerate(eps):
        for dst in eps[i+1:]:
            try:
                p = nx.shortest_path(G, src, dst, weight="weight")
                l = nx.shortest_path_length(G, src, dst, weight="weight")
                if l > best_l: best_l, best_p = l, p
            except nx.NetworkXNoPath:
                continue
    return best_p, best_l

def path_length(coords):
    """Euclidean sum along an ordered list of (r,c) coords."""
    pts = np.array(coords, dtype=float)
    return float(np.sum(np.linalg.norm(np.diff(pts, axis=0), axis=1)))


def improved_bridge(art_mask, skel, cath_path, n_direction_pts=10):
    """
    Better artifact bridge:

    1. Find which skeleton endpoint is nearest the artifact (entry).
    2. Estimate the catheter direction from the last n_direction_pts
       skeleton points before the entry endpoint.
    3. Cast a ray from the entry in that direction through the artifact
       to find the exit point.
    4. Also include the small gap between the skeleton endpoint and the
       nearest artifact edge.

    Returns (entry, exit_pt, total_bridge_px)
    """
    art_pts = np.argwhere(art_mask)
    if len(art_pts) == 0:
        warnings.warn("Artifact mask empty — bridge = 0.")
        return None, None, None, None, 0.0

    eps = get_endpoints(build_graph(skel))
    if not eps:
        return None, None, None, None, 0.0

    ep_arr = np.array(eps)

    # ── 1. Entry = skeleton endpoint closest to artifact ──────────────────
    d = np.linalg.norm(ep_arr[:, None] - art_pts[None, :], axis=2)
    ei, ai = np.unravel_index(d.argmin(), d.shape)
    entry        = tuple(ep_arr[ei])          # skeleton endpoint
    art_near_pt  = tuple(art_pts[ai])         # nearest artifact pixel to entry

    # ── 2. Direction from the last n points of the traced catheter path ───
    # Use the portion of the path near the valve end
    if len(cath_path) >= n_direction_pts:
        # figure out which end of cath_path is closest to entry
        if np.linalg.norm(np.array(cath_path[0]) - np.array(entry)) < \
           np.linalg.norm(np.array(cath_path[-1]) - np.array(entry)):
            tail = cath_path[:n_direction_pts]
        else:
            tail = cath_path[-n_direction_pts:]
        tail = np.array(tail, dtype=float)
        direction = tail[-1] - tail[0]
    else:
        tail      = np.array(cath_path, dtype=float)
        direction = tail[-1] - tail[0]

    norm = np.linalg.norm(direction)
    if norm == 0:
        direction = np.array([1.0, 0.0])
    else:
        direction = direction / norm

    # ── 3. Ray-cast through artifact to find exit ─────────────────────────
    # Project all artifact pixels onto the ray from entry
    entry_arr  = np.array(entry, dtype=float)
    projections = (art_pts - entry_arr) @ direction   # signed distance along ray
    # exit = artifact pixel with the largest positive projection
    pos_mask = projections > 0
    if pos_mask.any():
        exit_idx = np.argmax(projections * pos_mask)
    else:
        # fallback: farthest artifact pixel
        exit_idx = np.argmax(np.linalg.norm(art_pts - entry_arr, axis=1))
    exit_pt = tuple(art_pts[exit_idx])

    # ── 4. Gap between skeleton endpoint and artifact near edge ──────────
    gap_len = float(np.linalg.norm(np.array(art_near_pt) - entry_arr))

    # ── 5. Bridge = gap + straight line through artifact ─────────────────
    art_segment = float(np.linalg.norm(np.array(exit_pt) - np.array(art_near_pt)))
    total_bridge = gap_len + art_segment

    return entry, art_near_pt, exit_pt, direction, total_bridge


# ── pipeline ─────────────────────────────────────────────────────────────────

cath = to_2d_bool(predicted_catheter_mask, "catheter_mask")
art  = to_2d_bool(artifact_data,           "artifact_mask")

cath_clean = keep_largest(binary_fill_holes(cath))
skel       = skeletonize(cath_clean)

G   = build_graph(skel)
eps = get_endpoints(G)

if len(eps) < 2:
    pts  = np.argwhere(skel)
    dm   = np.linalg.norm(pts[:, None] - pts[None, :], axis=2)
    i, j = np.unravel_index(dm.argmax(), dm.shape)
    eps  = [tuple(pts[i]), tuple(pts[j])]

cath_path, _ = trace_longest(G, eps)

# Recompute visible length as true Euclidean path sum (more accurate than graph weight)
visible_px = path_length(cath_path)

# Identify tip vs valve end
art_props = regionprops(art.astype(np.uint8))
if art_props:
    art_c  = np.array(art_props[0].centroid)
    ep_arr = np.array(eps)
    vi     = int(np.linalg.norm(ep_arr - art_c, axis=1).argmin())
    tip, valve_end = eps[1 - vi], eps[vi]
else:
    tip, valve_end = cath_path[0], cath_path[-1]

entry, art_near_pt, exit_pt, direction, bridge_px = improved_bridge(
    art, skel, cath_path, n_direction_pts=15)

# Fallback if bridge failed
if exit_pt is None:
    entry, exit_pt, bridge_px = valve_end, valve_end, 0.0
    art_near_pt = valve_end

visible_mm = visible_px * PIXEL_SPACING_MM
hidden_mm  = bridge_px  * PIXEL_SPACING_MM
total_px   = visible_px + bridge_px
total_mm   = total_px   * PIXEL_SPACING_MM


# ── report ───────────────────────────────────────────────────────────────────

sep = "-" * 50
print(f"\n{sep}")
print("  CATHETER LENGTH ESTIMATION REPORT")
print(sep)
print(f"  Pixel spacing            : {PIXEL_SPACING_MM:.4f} mm/px")
print(sep)
print(f"  Visible skeleton length  : {visible_px:8.2f} px  ->  {visible_mm:7.2f} mm")
print(f"  Hidden (artifact) bridge : {bridge_px:8.2f} px  ->  {hidden_mm:7.2f} mm")
print(sep)
print(f"  TOTAL catheter length    : {total_px:8.2f} px  ->  {total_mm:7.2f} mm")
print(sep)
print(f"  Catheter tip   (row,col) : {tip}")
print(f"  Valve location (row,col) : {exit_pt}\n")


# ── visualisation ─────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor="#0d1117")
fig.suptitle("Catheter Length Estimation", color="white", fontsize=15, fontweight="bold")
for ax, t in zip(axes, ["Inputs", "Skeleton + Path", "Full Estimate"]):
    ax.set_facecolor("#0d1117"); ax.set_title(t, color="#e6edf3"); ax.axis("off")

# Panel 1
rgb = np.zeros((*cath.shape, 3))
rgb[cath] = [0.2, 0.6, 1.0]
rgb[art]  = [1.0, 0.4, 0.1]
axes[0].imshow(rgb)
axes[0].legend(handles=[
    mpatches.Patch(color=(0.2,0.6,1.0), label="Catheter mask"),
    mpatches.Patch(color=(1.0,0.4,0.1), label="Artifact mask"),
], loc="lower left", fontsize=8, labelcolor="white", facecolor="#1c2128", framealpha=0.5)

# Panel 2
rgb2 = np.zeros((*cath.shape, 3))
rgb2[cath_clean] = [0.12, 0.12, 0.12]
rgb2[skel]       = [0.9,  0.9,  0.2]
axes[1].imshow(rgb2)
if cath_path:
    axes[1].plot([p[1] for p in cath_path], [p[0] for p in cath_path], color="#00e5ff", lw=1.5)
axes[1].scatter(tip[1],       tip[0],       color="#00ff88", s=80, zorder=5)
axes[1].scatter(valve_end[1], valve_end[0], color="#ff4d6d", s=80, zorder=5)
axes[1].legend(handles=[
    Line2D([0],[0], color="#e6e620", lw=2, label="Skeleton"),
    Line2D([0],[0], color="#00e5ff", lw=2, label="Traced path"),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#00ff88', ms=8, label="Tip"),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#ff4d6d', ms=8, label="Valve end"),
], loc="lower left", fontsize=8, labelcolor="white", facecolor="#1c2128", framealpha=0.5)

# Panel 3
rgb3 = np.zeros((*cath.shape, 3))
rgb3[cath] = [0.1, 0.3, 0.5]
rgb3[art]  = [0.5, 0.2, 0.05]
axes[2].imshow(rgb3)
if cath_path:
    axes[2].plot([p[1] for p in cath_path], [p[0] for p in cath_path],
                 color="#00e5ff", lw=2.5, label=f"Visible {visible_mm:.1f} mm")

# Draw bridge: gap (entry -> art_near_pt) + artifact segment (art_near_pt -> exit_pt)
axes[2].plot([entry[1], art_near_pt[1]], [entry[0], art_near_pt[0]],
             color="#ff8800", lw=2, ls=":", label=f"Gap to artifact")
axes[2].plot([art_near_pt[1], exit_pt[1]], [art_near_pt[0], exit_pt[0]],
             color="#ffaa00", lw=2.5, ls="--", label=f"Hidden {hidden_mm:.1f} mm")

axes[2].scatter(tip[1],     tip[0],     color="#00ff88", s=100, zorder=6, label="Tip")
axes[2].scatter(exit_pt[1], exit_pt[0], color="#ff4d6d", s=100, zorder=6, label="Valve")
axes[2].set_xlabel(f"Total: {total_mm:.1f} mm  ({total_px:.1f} px)", color="#e6edf3", fontsize=10)
axes[2].legend(loc="lower left", fontsize=8, labelcolor="white", facecolor="#1c2128", framealpha=0.5)

plt.tight_layout()
plt.show()

[warn] 'catheter_mask' is 3-D; using slice 81
[warn] 'artifact_mask' is 3-D; using slice 102

--------------------------------------------------
  CATHETER LENGTH ESTIMATION REPORT
--------------------------------------------------
  Pixel spacing            : 1.0000 mm/px
--------------------------------------------------
  Visible skeleton length  :     5.66 px  ->     5.66 mm
  Hidden (artifact) bridge :    40.26 px  ->    40.26 mm
--------------------------------------------------
  TOTAL catheter length    :    45.92 px  ->    45.92 mm
--------------------------------------------------
  Catheter tip   (row,col) : (129, 143)
  Valve location (row,col) : (np.int64(159), np.int64(177))



/var/folders/0b/7bk2sbg51_v6nk5g8vs23vp40000gn/T/ipykernel_6979/1193089385.py:263: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
